# Entrenamiento y evaluación de modelos para predecir la variable **default** 

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path().resolve().parent.parent / "data/data-09-2025"
data_file = "cleaned_data_default.parquet"

df = pd.read_parquet(DATA_DIR / data_file)
df.head()

,plazo,vinculacion,v_cuota,v_prestamo,s_capital,s_intereses,aportes,garantias,valorgarantia,ctasahorros,...,actividadeconomica,estado_cliente,departamento,sexo,curtotalingresos,curtotalegresos,intestrato,actualizacion,default,puntaje_data
n_credito,,,,,,,,,,,,,,,,,,,,,
003-002-0125852-7,1827,8103,356849.0,15000000.0,12923538.0,123855,7741255,1,7741255,33042953.0,...,asalariados,1,antioquia,0,4597000.0,1500000.0,5.0,1,0,795.0
004-002-0068475-5,1826,1434,2650409.0,100460000.0,31911361.0,263265,4601706,1,4601706,3791115.0,...,asalariados,1,antioquia,0,4597000.0,650000.0,5.0,1,0,836.0
003-002-0122592-9,1826,573,791482.0,30000000.0,23844684.0,261477,530431,1,530431,94435.0,...,asalariados,1,antioquia,0,4400000.0,2000000.0,4.0,0,1,709.0
006-002-0023879-0,2922,1902,2860501.0,176000000.0,113842595.0,1008570,3023534,2,320385440,54841.0,...,educacion_basica_secundaria,1,antioquia,0,22020000.0,1500000.0,4.0,1,0,733.0
006-002-0026159-4,2557,1902,987637.0,50300000.0,38521256.0,317167,1023082,2,320385440,54841.0,...,educacion_basica_secundaria,1,antioquia,0,22020000.0,1500000.0,4.0,1,0,695.0


In [2]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12850 entries, 003-002-0125852-7 to 003-002-0119478-4
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   plazo               12850 non-null  int64   
 1   vinculacion         12850 non-null  int64   
 2   v_cuota             12850 non-null  float64 
 3   v_prestamo          12850 non-null  float64 
 4   s_capital           12850 non-null  float64 
 5   s_intereses         12850 non-null  int64   
 6   aportes             12850 non-null  int64   
 7   garantias           12850 non-null  int64   
 8   valorgarantia       12850 non-null  int64   
 9   ctasahorros         12850 non-null  float64 
 10  edad                12850 non-null  float64 
 11  tipoasociado        12850 non-null  int64   
 12  actividadeconomica  12850 non-null  category
 13  estado_cliente      12850 non-null  int64   
 14  departamento        12850 non-null  category
 15  sexo         

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["default", "s_capital"]) 
# s_capital está muy correlacionada con v_prestamo

y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (10280, 20)
Testing set size: (2570, 20)


In [4]:
from sklearn.metrics import classification_report, confusion_matrix


def print_resultados(model, X_test, y_test):
    print("reporte de clasificación:")
    print(classification_report(y_test, model.predict(X_test)), "\n")
    print("matriz de confusión:")
    print(confusion_matrix(y_test, model.predict(X_test)), "\n")
    feature_importances = pd.Series(model.feature_importances_, index=X_train.columns)
    feature_importances.sort_values(ascending=False, inplace=True)
    print("10 características más importantes:")
    print(feature_importances.head(10))
    return

# Modelos sin sintonizar

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

model = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    verbose=0,
    seed=1,
    feature_fraction_seed=1,
    bagging_seed=1,
    drop_seed=1,
    force_col_wise=True,
    deterministic=True,
    )

train_x, val_x, train_y, val_y = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
)

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

model.fit(
    train_x,
    train_y,
    categorical_feature=cat_vars,
    eval_set=[(val_x, val_y)],
    eval_metric="logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=20)],
)

train_score = f1_score(y_train, model.predict(X_train))
test_score = f1_score(y_test, model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.172171
Train score: 0.89
Test score: 0.76


In [12]:
print_resultados(model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.93      0.95      2125
           1       0.71      0.83      0.76       445

    accuracy                           0.91      2570
   macro avg       0.84      0.88      0.85      2570
weighted avg       0.92      0.91      0.91      2570
 

matriz de confusión:
[[1972  153]
 [  75  370]] 

10 características más importantes:
s_intereses         432
vinculacion         356
valorgarantia       272
puntaje_data        254
curtotalingresos    241
ctasahorros         235
v_cuota             199
plazo               181
v_prestamo          177
aportes             173
dtype: int32


In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from xgboost import XGBClassifier

te = TargetEncoder(random_state=1)

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

train_x, val_x, train_y, val_y = train_test_split(
    X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
)

model = XGBClassifier(
    objective="binary:logistic",
    grow_policy="lossguide",
    tree_method="hist",
    early_stopping_rounds=20,
    random_state=1,
    seed=1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    n_jobs=1,
    subsample=1.0,
    colsample_bytree=1.0,
)

model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

train_score = f1_score(y_train, model.predict(X_train_processed))
test_score = f1_score(y_test, model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.91
Test score: 0.77


In [41]:
print_resultados(model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      2125
           1       0.73      0.82      0.77       445

    accuracy                           0.91      2570
   macro avg       0.84      0.88      0.86      2570
weighted avg       0.92      0.91      0.92      2570
 

matriz de confusión:
[[1988  137]
 [  82  363]] 

10 características más importantes:
actualizacion       0.519547
estado_cliente      0.090687
tipoasociado        0.054463
garantias           0.036741
edad                0.032676
v_prestamo          0.026790
puntaje_data        0.026610
curtotalingresos    0.026059
departamento        0.023167
s_intereses         0.020663
dtype: float32


In [47]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    random_state=1, 
    max_depth=6
    )

model.fit(X_train_processed, y_train)
train_score = f1_score(y_train, model.predict(X_train_processed))
test_score = f1_score(y_test, model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.74
Test score: 0.68


In [48]:
print_resultados(model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.93      0.94      0.94      2125
           1       0.70      0.67      0.68       445

    accuracy                           0.89      2570
   macro avg       0.81      0.80      0.81      2570
weighted avg       0.89      0.89      0.89      2570
 

matriz de confusión:
[[1997  128]
 [ 149  296]] 

10 características más importantes:
actualizacion       0.249137
tipoasociado        0.226954
edad                0.141978
garantias           0.107553
v_prestamo          0.076327
aportes             0.063912
curtotalingresos    0.057202
s_intereses         0.032332
puntaje_data        0.028313
valorgarantia       0.009136
dtype: float64


# Sintonización de modelos con Optuna

### LightGBM

In [ ]:
import optuna
from optuna.samplers import TPESampler


def objective(trial):
    param = {
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 50, 500),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

    train_x, val_x, train_y, val_y = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LGBMClassifier(
        **param,
        objective="binary",
        verbose=-1,
        seed=1,
        feature_fraction_seed=1,
        bagging_seed=1,
        drop_seed=1,
        force_col_wise=True,
        deterministic=True,
    )

    model.fit(
        train_x,
        train_y,
        categorical_feature=categorical_features,
        eval_set=[(val_x, val_y)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=100, n_jobs=1)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

lgb_model = LGBMClassifier(
    **study.best_trial.params,
    verbose=-1,
    seed=1,
    feature_fraction_seed=1,
    bagging_seed=1,
    drop_seed=1,
    force_col_wise=True,
    deterministic=True,
)

categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

lgb_model.fit(
    X_train, 
    y_train,
    categorical_feature=categorical_features,
    )

lgb_params = study.best_trial.params

train_score = f1_score(y_train, lgb_model.predict(X_train))
test_score = f1_score(y_test, lgb_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-12 08:16:42,275] A new study created in memory with name: no-name-894b8314-6a2b-4583-b049-3dd9f7fe9f76
[I 2026-03-12 08:16:42,535] Trial 0 finished with value: 0.8165517241379311 and parameters: {'lambda_l1': 2.348881295853308e-05, 'lambda_l2': 3.6010467344475403, 'num_leaves': 380, 'feature_fraction': 0.759195090518222, 'bagging_fraction': 0.4936111842654619, 'bagging_freq': 2, 'min_child_samples': 10, 'learning_rate': 0.19030368381735815, 'colsample_bytree': 0.8005575058716043, 'scale_pos_weight': 7.372653200164409}. Best is trial 0 with value: 0.8165517241379311.
[I 2026-03-12 08:16:42,674] Trial 1 finished with value: 0.806970509383378 and parameters: {'lambda_l1': 1.5320059381854043e-08, 'lambda_l2': 5.360294728728285, 'num_leaves': 425, 'feature_fraction': 0.5274034664069657, 'bagging_fraction': 0.5090949803242604, 'bagging_freq': 2, 'min_child_samples': 34, 'learning_rate': 0.05958389350068958, 'colsample_bytree': 0.7159725093210578, 'scale_pos_weight': 3.621062261782

Best trial: 0.839 with params: {'lambda_l1': 0.001341429055420671, 'lambda_l2': 6.360121113121276e-08, 'num_leaves': 172, 'feature_fraction': 0.7568139135285814, 'bagging_fraction': 0.7861571703882632, 'bagging_freq': 7, 'min_child_samples': 45, 'learning_rate': 0.11657126424443101, 'colsample_bytree': 0.9250988187445204, 'scale_pos_weight': 4.284863340608763}
Train score: 0.99
Test score: 0.80


In [63]:
print_resultados(lgb_model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96      2125
           1       0.81      0.79      0.80       445

    accuracy                           0.93      2570
   macro avg       0.88      0.87      0.88      2570
weighted avg       0.93      0.93      0.93      2570
 

matriz de confusión:
[[2041   84]
 [  94  351]] 

10 características más importantes:
s_intereses         1650
ctasahorros         1309
vinculacion         1301
puntaje_data        1251
valorgarantia       1179
v_cuota             1160
curtotalingresos    1115
aportes              893
v_prestamo           855
edad                 817
dtype: int32


### XGBoost

In [ ]:
def objective(trial):
    te = TargetEncoder(random_state=1)

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 5e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1e2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1e2, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    model = XGBClassifier(
        objective="binary:logistic",
        grow_policy="lossguide",
        tree_method="hist",
        early_stopping_rounds=20,
        random_state=1,
        seed=1,
        n_jobs=1,
        subsample=1.0,
        colsample_bytree=1.0,
    )

    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=100, n_jobs=1)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

xgb_model = XGBClassifier(
    **study.best_trial.params,
    objective="binary:logistic",
    grow_policy="lossguide",
    tree_method="hist",
    random_state=1,
    seed=1,
    n_jobs=1,
    subsample=1.0,
    colsample_bytree=1.0,
)

te = TargetEncoder(random_state=1)

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

xgb_model.fit(X_train_processed, y_train)
xgb_params = study.best_trial.params

train_score = f1_score(y_train, xgb_model.predict(X_train_processed))
test_score = f1_score(y_test, xgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-12 08:23:29,711] A new study created in memory with name: no-name-6065ae80-7600-4ad7-8111-a46f4217cb60
[I 2026-03-12 08:23:29,848] Trial 0 finished with value: 0.8 and parameters: {'max_depth': 7, 'learning_rate': 0.4123206532618726, 'n_estimators': 380, 'reg_alpha': 0.9846738873614566, 'reg_lambda': 0.006026889128682512, 'max_delta_step': 1, 'gamma': 0.11616722433639892, 'min_child_weight': 10, 'scale_pos_weight': 6.41003510568888}. Best is trial 0 with value: 0.8.
[I 2026-03-12 08:23:29,976] Trial 1 finished with value: 0.8 and parameters: {'max_depth': 12, 'learning_rate': 0.01083858126934475, 'n_estimators': 487, 'reg_alpha': 14.528246637516036, 'reg_lambda': 0.011526449540315618, 'max_delta_step': 2, 'gamma': 0.36680901970686763, 'min_child_weight': 6, 'scale_pos_weight': 5.72280788469014}. Best is trial 0 with value: 0.8.
[I 2026-03-12 08:23:30,098] Trial 2 finished with value: 0.8 and parameters: {'max_depth': 8, 'learning_rate': 0.03124565071260872, 'n_estimators': 3

Best trial: 0.800 with params: {'max_depth': 7, 'learning_rate': 0.4123206532618726, 'n_estimators': 380, 'reg_alpha': 0.9846738873614566, 'reg_lambda': 0.006026889128682512, 'max_delta_step': 1, 'gamma': 0.11616722433639892, 'min_child_weight': 10, 'scale_pos_weight': 6.41003510568888}
Train score: 1.00
Test score: 0.78


In [68]:
print_resultados(xgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.95      0.95      2125
           1       0.77      0.79      0.78       445

    accuracy                           0.92      2570
   macro avg       0.86      0.87      0.87      2570
weighted avg       0.92      0.92      0.92      2570
 

matriz de confusión:
[[2019  106]
 [  93  352]] 

10 características más importantes:
actualizacion       0.616053
estado_cliente      0.083982
tipoasociado        0.042686
garantias           0.029885
edad                0.024317
curtotalingresos    0.022422
v_prestamo          0.020810
aportes             0.019681
puntaje_data        0.019149
ctasahorros         0.019127
dtype: float32


### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier


def objective(trial):
    te = TargetEncoder(random_state=1)

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = RandomForestClassifier(class_weight="balanced", random_state=1)

    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
    }

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=100, n_jobs=1)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

rf_model = RandomForestClassifier(
    **study.best_trial.params,
    class_weight="balanced",
    random_state=1,
)

te = TargetEncoder(random_state=1)

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

rf_model.fit(X_train_processed, y_train)
rf_params = study.best_trial.params

train_score = f1_score(y_train, rf_model.predict(X_train_processed))
test_score = f1_score(y_test, rf_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-12 08:30:43,465] A new study created in memory with name: no-name-b3e55816-81fd-4f9b-a89c-7c7d1592048b
[I 2026-03-12 08:30:44,572] Trial 0 finished with value: 0.759075907590759 and parameters: {'n_estimators': 218, 'max_depth': 15, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.759075907590759.
[I 2026-03-12 08:30:45,648] Trial 1 finished with value: 0.759075907590759 and parameters: {'n_estimators': 440, 'max_depth': 10, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.759075907590759.
[I 2026-03-12 08:30:46,707] Trial 2 finished with value: 0.759075907590759 and parameters: {'n_estimators': 132, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 11, 'max_features': None}. Best is trial 0 with value: 0.759075907590759.
[I 2026-03-12 08:30:47,814] Trial 3 finished with value: 0.759075907590759 and parameters: {'n_estimators': 112, 'max_depth': 6, 'min_samp

Best trial: 0.759 with params: {'n_estimators': 218, 'max_depth': 15, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt'}
Train score: 0.83
Test score: 0.75


In [70]:
print_resultados(rf_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.93      0.94      2125
           1       0.70      0.82      0.75       445

    accuracy                           0.91      2570
   macro avg       0.83      0.87      0.85      2570
weighted avg       0.91      0.91      0.91      2570
 

matriz de confusión:
[[1966  159]
 [  81  364]] 

10 características más importantes:
actualizacion       0.212796
tipoasociado        0.145131
garantias           0.129773
curtotalingresos    0.076062
v_prestamo          0.072643
edad                0.071129
puntaje_data        0.065378
estado_cliente      0.056453
valorgarantia       0.045533
s_intereses         0.037997
dtype: float64


## Logistic Regression

In [102]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PowerTransformer


def objective(trial):
    te = TargetEncoder(random_state=1)
    pt = PowerTransformer()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()
    num_vars = [
        "plazo",
        "v_cuota",
        "v_prestamo",
        "s_intereses",
        "aportes",
        "valorgarantia",
        "ctasahorros",
        "edad",
        "curtotalingresos",
        "curtotalegresos",
        "puntaje_data",
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
            ("power_transformer", pt, num_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LogisticRegression(
        class_weight="balanced", 
        solver="saga", 
        max_iter=10000, 
        random_state=1,
        n_jobs=1,
    )

    param = {
        "C": trial.suggest_float("C", 1e-3, 1e3, log=True),
        # "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "penalty": trial.suggest_categorical("penalty", ["l1", "l2", None]),
    }

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=20, n_jobs=1)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

lr_model = LogisticRegression(
    **study.best_trial.params,
    class_weight="balanced",
    solver="saga",
    max_iter=10000,
    random_state=1,
    n_jobs=1,
)

te = TargetEncoder(random_state=1)
pt = PowerTransformer()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

num_vars = [
    "plazo",
    "v_cuota",
    "v_prestamo",
    "s_intereses",
    "aportes",
    "valorgarantia",
    "ctasahorros",
    "edad",
    "curtotalingresos",
    "curtotalegresos",
    "puntaje_data",
]

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
        ("power_transformer", pt, num_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

lr_model.fit(X_train_processed, y_train)
lr_params = lr_model.get_params()

train_score = f1_score(y_train, lr_model.predict(X_train_processed))
test_score = f1_score(y_test, lr_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-12 09:18:24,619] A new study created in memory with name: no-name-1c879f9a-cad0-4cb1-8108-e4949883f1d5
[I 2026-03-12 09:18:34,323] Trial 0 finished with value: 0.5032258064516129 and parameters: {'C': 0.1767016940294795, 'penalty': 'l1'}. Best is trial 0 with value: 0.5032258064516129.
[I 2026-03-12 09:18:44,546] Trial 1 finished with value: 0.5032258064516129 and parameters: {'C': 0.008632008168602538, 'penalty': None}. Best is trial 0 with value: 0.5032258064516129.
[I 2026-03-12 09:18:54,136] Trial 2 finished with value: 0.5032258064516129 and parameters: {'C': 4.0428727350273315, 'penalty': None}. Best is trial 0 with value: 0.5032258064516129.
[I 2026-03-12 09:19:03,514] Trial 3 finished with value: 0.5032258064516129 and parameters: {'C': 98.77700294007911, 'penalty': 'l1'}. Best is trial 0 with value: 0.5032258064516129.
[I 2026-03-12 09:19:13,722] Trial 4 finished with value: 0.5032258064516129 and parameters: {'C': 0.06690421166498801, 'penalty': 'l1'}. Best is tria

Best trial: 0.503 with params: {'C': 0.1767016940294795, 'penalty': 'l1'}
Train score: 0.49
Test score: 0.50


In [103]:
print(classification_report(lr_model.predict(X_test_processed), y_test))
print(confusion_matrix(lr_model.predict(X_test_processed), y_test))

              precision    recall  f1-score   support

           0       0.69      0.95      0.80      1553
           1       0.81      0.36      0.50      1017

    accuracy                           0.71      2570
   macro avg       0.75      0.65      0.65      2570
weighted avg       0.74      0.71      0.68      2570

[[1470   83]
 [ 655  362]]


In [104]:
pesos = pd.Series(lr_model.coef_[0], index=X_train.columns.to_list())
pesos.loc["intercept"] = lr_model.intercept_[0]
pesos.sort_values(ascending=False)

aportes               0.049587
curtotalingresos      0.026842
v_cuota               0.007761
curtotalegresos       0.003408
actualizacion         0.001823
intercept             0.001688
plazo                 0.001149
vinculacion           0.000099
departamento         -0.000008
intestrato           -0.007392
s_intereses          -0.015573
edad                 -0.017203
actividadeconomica   -0.022579
v_prestamo           -0.026316
tipoasociado         -0.027238
sexo                 -0.037633
garantias            -0.042549
puntaje_data         -0.042694
valorgarantia        -0.045408
estado_cliente       -0.051635
ctasahorros          -0.078112
dtype: float64

# Modelos con Skrub

### LightGBM

In [72]:
def print_resultados_skrub(model, X_test, y_test):
    print("reporte de clasificación:")
    print(classification_report(y_test, model.predict(X_test)), "\n")
    print("matriz de confusión:")
    print(confusion_matrix(y_test, model.predict(X_test)), "\n")
    feature_importances = pd.Series(
        model.feature_importances_, index=X_train_processed.columns.to_list()
    )
    feature_importances.sort_values(ascending=False, inplace=True)
    print("10 características más importantes:")
    print(feature_importances.head(10))
    return

In [82]:
import warnings

from skrub import TableVectorizer

warnings.filterwarnings("ignore")


def objective(trial):
    param = {
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 50, 500),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    vectorizer = TableVectorizer(n_jobs=1)

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LGBMClassifier(
        **param,
        objective="binary",
        verbose=-1,
        seed=1,
        feature_fraction_seed=1,
        bagging_seed=1,
        drop_seed=1,
        force_col_wise=True,
        deterministic=True,
    )

    model.fit(
        train_x,
        train_y,
        eval_set=[(val_x, val_y)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=100, n_jobs=1)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

vectorizer = TableVectorizer(n_jobs=1)

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_lgb_model = LGBMClassifier(
    **study.best_trial.params,
    objective="binary",
    force_col_wise=True,
    random_state=1,
    seed=1,
    feature_fraction_seed=1,
    bagging_seed=1,
    drop_seed=1,
    deterministic=True,
)

sk_lgb_model.fit(X_train_processed, y_train)
sk_lgb_params = study.best_trial.params

train_score = f1_score(y_train, sk_lgb_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_lgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-12 08:48:11,408] A new study created in memory with name: no-name-16337d5a-15b2-482a-8baa-18c22211c0fb


[I 2026-03-12 08:48:12,185] Trial 0 finished with value: 0.8093278463648834 and parameters: {'lambda_l1': 2.348881295853308e-05, 'lambda_l2': 3.6010467344475403, 'num_leaves': 380, 'feature_fraction': 0.759195090518222, 'bagging_fraction': 0.4936111842654619, 'bagging_freq': 2, 'min_child_samples': 10, 'learning_rate': 0.19030368381735815, 'colsample_bytree': 0.8005575058716043, 'scale_pos_weight': 7.372653200164409}. Best is trial 0 with value: 0.8093278463648834.
[I 2026-03-12 08:48:12,751] Trial 1 finished with value: 0.8179347826086957 and parameters: {'lambda_l1': 1.5320059381854043e-08, 'lambda_l2': 5.360294728728285, 'num_leaves': 425, 'feature_fraction': 0.5274034664069657, 'bagging_fraction': 0.5090949803242604, 'bagging_freq': 2, 'min_child_samples': 34, 'learning_rate': 0.05958389350068958, 'colsample_bytree': 0.7159725093210578, 'scale_pos_weight': 3.6210622617823773}. Best is trial 1 with value: 0.8179347826086957.
[I 2026-03-12 08:48:13,430] Trial 2 finished with value: 0

Best trial: 0.834 with params: {'lambda_l1': 0.0038470497565927138, 'lambda_l2': 0.0002360662880553757, 'num_leaves': 290, 'feature_fraction': 0.6385589921227967, 'bagging_fraction': 0.7892547618020631, 'bagging_freq': 4, 'min_child_samples': 19, 'learning_rate': 0.08814326732935757, 'colsample_bytree': 0.9292614361673384, 'scale_pos_weight': 9.94071374011233}
Train score: 1.00
Test score: 0.79


In [83]:
print_resultados_skrub(sk_lgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96      2125
           1       0.79      0.79      0.79       445

    accuracy                           0.93      2570
   macro avg       0.87      0.87      0.87      2570
weighted avg       0.93      0.93      0.93      2570
 

matriz de confusión:
[[2034   91]
 [  94  351]] 

10 características más importantes:
s_intereses         2830
ctasahorros         2754
puntaje_data        2549
vinculacion         2423
curtotalingresos    2150
valorgarantia       2091
v_cuota             1999
aportes             1935
v_prestamo          1791
edad                1551
dtype: int32


### Skrub con modelo por defecto

In [86]:
from skrub import tabular_pipeline


def objective(trial):
    param = {
        "histgradientboostingclassifier__learning_rate": trial.suggest_float(
            "histgradientboostingclassifier__learning_rate", 0.01, 0.2
        ),  # noqa: E501
        "histgradientboostingclassifier__max_iter": trial.suggest_int(
            "histgradientboostingclassifier__max_iter", 100, 500
        ),  # noqa: E501
        "histgradientboostingclassifier__max_leaf_nodes": trial.suggest_int(
            "histgradientboostingclassifier__max_leaf_nodes", 20, 100
        ),  # noqa: E501
        "histgradientboostingclassifier__min_samples_leaf": trial.suggest_int(
            "histgradientboostingclassifier__min_samples_leaf", 1, 10
        ),  # noqa: E501
        "histgradientboostingclassifier__l2_regularization": trial.suggest_float(
            "histgradientboostingclassifier__l2_regularization", 1e-3, 10.0, log=True
        ),  # noqa: E501
        "histgradientboostingclassifier__early_stopping": trial.suggest_categorical(
            "histgradientboostingclassifier__early_stopping", [True, False]
        ),  # noqa: E501
    }

    train_x, val_x, train_y, val_y = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = tabular_pipeline(estimator="classifier", n_jobs=1)

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=100, n_jobs=1)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_params = study.best_trial.params
sk_params = {
    k.replace("histgradientboostingclassifier__", ""): v for k, v in sk_params.items()
}

sk_model = tabular_pipeline(estimator="classifier", n_jobs=1)
sk_model["histgradientboostingclassifier"].set_params(**sk_params)
sk_model.fit(X_train, y_train)
sk_params = sk_model.get_params()

train_score = f1_score(y_train, sk_model.predict(X_train))
test_score = f1_score(y_test, sk_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-12 08:55:39,024] A new study created in memory with name: no-name-8937e134-0f6c-4875-ab75-d34e3c978130
[I 2026-03-12 08:55:39,616] Trial 0 finished with value: 0.8085106382978723 and parameters: {'histgradientboostingclassifier__learning_rate': 0.08116262258099886, 'histgradientboostingclassifier__max_iter': 481, 'histgradientboostingclassifier__max_leaf_nodes': 79, 'histgradientboostingclassifier__min_samples_leaf': 6, 'histgradientboostingclassifier__l2_regularization': 0.004207988669606638, 'histgradientboostingclassifier__early_stopping': True}. Best is trial 0 with value: 0.8085106382978723.
[I 2026-03-12 08:55:40,231] Trial 1 finished with value: 0.803030303030303 and parameters: {'histgradientboostingclassifier__learning_rate': 0.1745734676972377, 'histgradientboostingclassifier__max_iter': 341, 'histgradientboostingclassifier__max_leaf_nodes': 77, 'histgradientboostingclassifier__min_samples_leaf': 1, 'histgradientboostingclassifier__l2_regularization': 7.57947995334

Best trial: 0.828 with params: {'histgradientboostingclassifier__learning_rate': 0.026813575389864702, 'histgradientboostingclassifier__max_iter': 178, 'histgradientboostingclassifier__max_leaf_nodes': 23, 'histgradientboostingclassifier__min_samples_leaf': 4, 'histgradientboostingclassifier__l2_regularization': 0.03586816498627549, 'histgradientboostingclassifier__early_stopping': False}
Train score: 0.84
Test score: 0.76


In [87]:
print("reporte de clasificación:")
print(classification_report(y_test, sk_model.predict(X_test)), "\n")
print("matriz de confusión:")
print(confusion_matrix(y_test, sk_model.predict(X_test)), "\n")

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95      2125
           1       0.83      0.70      0.76       445

    accuracy                           0.92      2570
   macro avg       0.88      0.83      0.86      2570
weighted avg       0.92      0.92      0.92      2570
 

matriz de confusión:
[[2061   64]
 [ 135  310]] 



### XGBoost

In [90]:
def objective(trial):
    vectorizer = TableVectorizer(n_jobs=1)

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 5e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1e2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1e2, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    model = XGBClassifier(
        objective="binary:logistic",
        grow_policy="lossguide",
        tree_method="hist",
        early_stopping_rounds=20,
        random_state=1,
        seed=1,
        n_jobs=1,
        subsample=1.0,
        colsample_bytree=1.0,
    )

    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=100, n_jobs=1)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_xgb_model = XGBClassifier(
    **study.best_trial.params,
    objective="binary:logistic",
    grow_policy="lossguide",
    tree_method="hist",
    random_state=1,
    seed=1,
    n_jobs=1,
    subsample=1.0,
    colsample_bytree=1.0,
)

vectorizer = TableVectorizer(n_jobs=1)

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_xgb_model.fit(X_train_processed, y_train)
sk_xgb_params = study.best_trial.params

train_score = f1_score(y_train, sk_xgb_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_xgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-12 09:00:29,908] A new study created in memory with name: no-name-7c02e925-3eca-4450-93ac-ae8e2dea6f96
[I 2026-03-12 09:00:30,607] Trial 0 finished with value: 0.7945205479452054 and parameters: {'max_depth': 7, 'learning_rate': 0.4123206532618726, 'n_estimators': 380, 'reg_alpha': 0.9846738873614566, 'reg_lambda': 0.006026889128682512, 'max_delta_step': 1, 'gamma': 0.11616722433639892, 'min_child_weight': 10, 'scale_pos_weight': 6.41003510568888}. Best is trial 0 with value: 0.7945205479452054.
[I 2026-03-12 09:00:31,269] Trial 1 finished with value: 0.8048048048048048 and parameters: {'max_depth': 12, 'learning_rate': 0.01083858126934475, 'n_estimators': 487, 'reg_alpha': 14.528246637516036, 'reg_lambda': 0.011526449540315618, 'max_delta_step': 2, 'gamma': 0.36680901970686763, 'min_child_weight': 6, 'scale_pos_weight': 5.72280788469014}. Best is trial 1 with value: 0.8048048048048048.
[I 2026-03-12 09:00:31,950] Trial 2 finished with value: 0.8072837632776935 and parameter

Best trial: 0.824 with params: {'max_depth': 9, 'learning_rate': 0.3403424732448265, 'n_estimators': 264, 'reg_alpha': 0.004526576334707321, 'reg_lambda': 0.0020915870931296735, 'max_delta_step': 5, 'gamma': 1.2979064782012437, 'min_child_weight': 9, 'scale_pos_weight': 3.1128719280880732}
Train score: 0.94
Test score: 0.78


In [91]:
print_resultados_skrub(sk_xgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      2125
           1       0.75      0.81      0.78       445

    accuracy                           0.92      2570
   macro avg       0.86      0.88      0.87      2570
weighted avg       0.92      0.92      0.92      2570
 

matriz de confusión:
[[2006  119]
 [  85  360]] 

10 características más importantes:
actualizacion            0.556569
ctasahorros              0.038553
tipoasociado             0.029064
valorgarantia            0.025249
garantias                0.023400
s_intereses              0.021439
actividadeconomica_00    0.020616
curtotalingresos         0.018901
vinculacion              0.017382
puntaje_data             0.016471
dtype: float32


### Random Forest

In [94]:
def objective(trial):
    vectorizer = TableVectorizer(n_jobs=1)

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = RandomForestClassifier(class_weight="balanced", random_state=1)

    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
    }

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=100, n_jobs=1)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_rf_model = RandomForestClassifier(
    **study.best_trial.params,
    class_weight="balanced",
    random_state=1,
)

vectorizer = TableVectorizer(n_jobs=1)

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_rf_model.fit(X_train_processed, y_train)
sk_rf_params = study.best_trial.params

train_score = f1_score(y_train, sk_rf_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_rf_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-12 09:06:10,280] A new study created in memory with name: no-name-ff9eee09-4438-49c8-b5f7-592d53605289
[I 2026-03-12 09:06:11,660] Trial 0 finished with value: 0.7325383304940375 and parameters: {'n_estimators': 218, 'max_depth': 15, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7325383304940375.
[I 2026-03-12 09:06:12,957] Trial 1 finished with value: 0.7327731092436974 and parameters: {'n_estimators': 440, 'max_depth': 10, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.7327731092436974.
[I 2026-03-12 09:06:14,257] Trial 2 finished with value: 0.7343485617597293 and parameters: {'n_estimators': 132, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 11, 'max_features': None}. Best is trial 2 with value: 0.7343485617597293.
[I 2026-03-12 09:06:15,672] Trial 3 finished with value: 0.7247863247863248 and parameters: {'n_estimators': 112, 'max_depth': 6, 'm

Best trial: 0.753 with params: {'n_estimators': 451, 'max_depth': 10, 'min_samples_split': 15, 'min_samples_leaf': 10, 'max_features': 'sqrt'}
Train score: 0.80
Test score: 0.74


In [95]:
print_resultados_skrub(sk_rf_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.92      0.94      2125
           1       0.68      0.82      0.74       445

    accuracy                           0.90      2570
   macro avg       0.82      0.87      0.84      2570
weighted avg       0.91      0.90      0.90      2570
 

matriz de confusión:
[[1949  176]
 [  78  367]] 

10 características más importantes:
actualizacion       0.206223
s_intereses         0.125862
ctasahorros         0.123315
curtotalingresos    0.090209
valorgarantia       0.066962
puntaje_data        0.064418
tipoasociado        0.063630
vinculacion         0.061994
aportes             0.048898
v_cuota             0.032498
dtype: float64


## Logistic Regression

In [108]:
def objective(trial):
    
    vectorizer = TableVectorizer(n_jobs=1)
    pt = PowerTransformer()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()
    num_vars = [
        "plazo",
        "v_cuota",
        "v_prestamo",
        "s_intereses",
        "aportes",
        "valorgarantia",
        "ctasahorros",
        "edad",
        "curtotalingresos",
        "curtotalegresos",
        "puntaje_data",
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("power_transformer", pt, num_vars),
            ("vectorizer", vectorizer, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LogisticRegression(
        class_weight="balanced", 
        solver="saga", 
        max_iter=10000, 
        random_state=1,
        n_jobs=1,
    )

    param = {
        "C": trial.suggest_float("C", 1e-3, 1e3, log=True),
        # "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "penalty": trial.suggest_categorical("penalty", ["l1", "l2", None]),
    }

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5, n_jobs=1)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_lr_model = LogisticRegression(
    **study.best_trial.params,
    class_weight="balanced",
    solver="saga",
    max_iter=10000,
    random_state=1,
    n_jobs=1,
)

vectorizer = TableVectorizer(n_jobs=1)
pt = PowerTransformer()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

num_vars = [
    "plazo",
    "v_cuota",
    "v_prestamo",
    "s_intereses",
    "aportes",
    "valorgarantia",
    "ctasahorros",
    "edad",
    "curtotalingresos",
    "curtotalegresos",
    "puntaje_data",
]

preprocessor = ColumnTransformer(
    transformers=[
        ("vectorizer", vectorizer, cat_vars),
        ("power_transformer", pt, num_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

sk_lr_model.fit(X_train_processed, y_train)
sk_lr_params = study.best_trial.params

train_score = f1_score(y_train, sk_lr_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_lr_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-12 09:29:28,824] A new study created in memory with name: no-name-f88f358d-8ac8-4c55-b83d-4951e4cfb14c
[I 2026-03-12 09:30:11,538] Trial 0 finished with value: 0.5009107468123861 and parameters: {'C': 0.1767016940294795, 'penalty': 'l1'}. Best is trial 0 with value: 0.5009107468123861.
[I 2026-03-12 09:30:51,983] Trial 1 finished with value: 0.5009107468123861 and parameters: {'C': 0.008632008168602538, 'penalty': None}. Best is trial 0 with value: 0.5009107468123861.
[I 2026-03-12 09:31:25,917] Trial 2 finished with value: 0.5009107468123861 and parameters: {'C': 4.0428727350273315, 'penalty': None}. Best is trial 0 with value: 0.5009107468123861.
[I 2026-03-12 09:31:59,445] Trial 3 finished with value: 0.5009107468123861 and parameters: {'C': 98.77700294007911, 'penalty': 'l1'}. Best is trial 0 with value: 0.5009107468123861.
[I 2026-03-12 09:32:39,609] Trial 4 finished with value: 0.5009107468123861 and parameters: {'C': 0.06690421166498801, 'penalty': 'l1'}. Best is tria

Best trial: 0.501 with params: {'C': 0.1767016940294795, 'penalty': 'l1'}
Train score: 0.49
Test score: 0.49


In [109]:
print(classification_report(sk_lr_model.predict(X_test_processed), y_test))
print(confusion_matrix(sk_lr_model.predict(X_test_processed), y_test))

              precision    recall  f1-score   support

           0       0.68      0.95      0.79      1533
           1       0.82      0.35      0.49      1037

    accuracy                           0.71      2570
   macro avg       0.75      0.65      0.64      2570
weighted avg       0.74      0.71      0.67      2570

[[1454   79]
 [ 671  366]]


In [110]:
pesos = pd.Series(sk_lr_model.coef_[0], index=preprocessor.get_feature_names_out())
pesos.loc["intercept"] = sk_lr_model.intercept_[0]
pesos.sort_values(ascending=False)

power_transformer__s_intereses       0.049541
remainder__tipoasociado              0.026723
power_transformer__plazo             0.007811
vectorizer__actividadeconomica_00    0.003317
remainder__estado_cliente            0.003188
                                       ...   
power_transformer__aportes          -0.042372
remainder__actualizacion            -0.042764
power_transformer__valorgarantia    -0.045229
power_transformer__puntaje_data     -0.051534
power_transformer__ctasahorros      -0.078017
Length: 71, dtype: float64

# Guardado de modelos y parámetos

In [111]:
import json

import joblib

MODELS_DIR = Path().resolve().parent / "models"
models = {
    "lgb": (lgb_model, lgb_params),
    "xgb": (xgb_model, xgb_params),
    "rf": (rf_model, rf_params),
    "lr": (lr_model, lr_params),
    "sk_lgb": (sk_lgb_model, sk_lgb_params),
    "sk_xgb": (sk_xgb_model, sk_xgb_params),
    "sk_rf": (sk_rf_model, sk_rf_params),
    "sk_lr": (sk_lr_model, sk_lr_params),
    }

# Guardar modelos en formato joblib y parámetros en formato JSON    

for name, (model, params) in models.items():
    joblib.dump(model, MODELS_DIR / f"{name}_model.joblib")
    with open(MODELS_DIR / f"{name}_params.json", "w") as f:
        json.dump(params, f)